# AU-AIR Object Detection

I run this code on Kaggle. With accelerator → **GPU T4 x2**, one epoch takes 20 min.
All env settings were done according to Kaggle requirements

## Step 1: Environment Setup

In [ ]:
import os, sys
from pathlib import Path

print('Contents of /kaggle/input/:')
for item in sorted(Path('/kaggle/input').iterdir()):
    print(f'  {item}')
    for sub in sorted(item.iterdir())[:5]:   # show first 5 files per dataset
        print(f'    {sub.name}')


In [ ]:
# FOR KAGGLE
# Auto-find annotations.json anywhere under /kaggle/input/
matches = list(Path('/kaggle/input').rglob('annotations.json'))
if not matches:
    raise FileNotFoundError('annotations.json not found under /kaggle/input/ — did you add the dataset?')

ANN_PATH     = matches[0]
KAGGLE_INPUT = ANN_PATH.parent          # folder that should contain annotations.json
IMG_DIR      = KAGGLE_INPUT / 'images'
PROJECT_ROOT = Path('/kaggle/working')
DATA_DIR     = KAGGLE_INPUT
YOLO_ROOT    = PROJECT_ROOT / 'auair_yolo'

print('Dataset found at :', KAGGLE_INPUT)
print('ANN_PATH         :', ANN_PATH)
print('IMG_DIR          :', IMG_DIR)
print('IMG_DIR exists   :', IMG_DIR.exists())


In [ ]:
# Install required packages on KAggle
import subprocess, sys

# Packages
packages = ['ultralytics>=8.2', 'sahi>=0.11']

for pkg in packages:
    print(f'Installing {pkg} ...')
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'  ERROR: {result.stderr[-300:]}')
    else:
        print(f'  OK')

print('\nAll packages installed.')


In [ ]:
import sys
print(sys.executable)

In [ ]:
import json
import os
import random
import shutil
from pathlib import Path
from collections import Counter, defaultdict
import yaml

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from pathlib import Path
from collections import Counter

# Verify installs
import ultralytics
import sahi
print(f"ultralytics version : {ultralytics.__version__}")
print(f"sahi version        : {sahi.__version__}")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## Step 2: Dataset Exploration

In [ ]:
with open(ANN_PATH) as f:
    raw = json.load(f)

CATEGORIES   = raw['categories']
annotations  = raw['annotations']
total_bboxes = sum(len(a['bbox']) for a in annotations)

print('Categories :', CATEGORIES)
print(f'Total images : {len(annotations):,}')
print(f'Total bboxes : {total_bboxes:,}')


### Image viz with Labels

In [ ]:
# Color palette (one per class)
PALETTE = [
    "#e6194b", "#3cb44b", "#ffe119", "#4363d8",
    "#f58231", "#911eb4", "#42d4f4", "#f032e6"
]

def draw_bboxes(ax, image, bboxes, categories, palette):
    ax.imshow(image)
    for bbox in bboxes:
        cls   = bbox["class"]
        left  = bbox["left"]
        top   = bbox["top"]
        w     = bbox["width"]
        h     = bbox["height"]
        color = palette[cls % len(palette)]
        rect  = patches.Rectangle(
            (left, top), w, h,
            linewidth=2, edgecolor=color, facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(
            left, top - 4, categories[cls],
            color="white", fontsize=7, fontweight="bold",
            bbox=dict(facecolor=color, alpha=0.8, pad=1, edgecolor="none")
        )
    ax.axis("off")

# 4 representative samples
rng = random.Random(SEED)
samples = rng.sample(annotations, 4)

fig, axes = plt.subplots(2, 2, figsize=(16, 9))
axes = axes.flatten()

for ax, ann in zip(axes, samples):
    img_path = IMG_DIR / ann["image_name"]
    img = Image.open(img_path).convert("RGB")
    draw_bboxes(ax, img, ann["bbox"], CATEGORIES, PALETTE)
    n_obj = len(ann["bbox"])
    ax.set_title(f"{ann['image_name']}  ({n_obj} objects)", fontsize=8)


plt.show()

* Important details from few viz: It labeled the cars in the parking area as a whole!!
* They are the provided ground truths from the AU-AIR annotation file. The parked cars at the top look grouped into one large Car box in one image that means the dataset annotation itself likely labeled the parked cluster coarsely.

SO:

what if the model below predicts several separate car boxes for these areas?
 * Unfortunately, the evaluator will compare model  predictions against the annotation file, not against what looks visually “more correct” to us.

What usually happens:

* one or more predicted boxes may have low IoU with the single large ground-truth box
* the large ground-truth box may remain unmatched
* model's individual detections can be counted as false positives
* recall and AP can drop even if the prediction is semantically better --> NOT FORGET THIS DETAILS


So for this assignment, for safety:
* optimize for matching the dataset’s annotation style
* not for “humanly better” labeling if it disagrees with the labels


Discuss it in report:
* the dataset contains some coarse annotations
* in parking areas, grouped vehicle labeling may penalize finer-grained detectors
* therefore evaluation may underestimate true qualitative performance

### Class Distribution

In [ ]:
class_counts = Counter()
for ann in annotations:
    for bbox in ann["bbox"]:
        class_counts[bbox["class"]] += 1

print(f"{'Class':<12} {'Count':>8} {'Percent':>8}")
print("-" * 32)
for cls_id, count in sorted(class_counts.items(), key=lambda x: -x[1]):
    pct = 100 * count / total_bboxes
    print(f"{CATEGORIES[cls_id]:<12} {count:>8,} {pct:>7.2f}%")

In [ ]:
# Look at class distribution by using bar chart
cls_ids   = sorted(class_counts.keys())
cls_names = [CATEGORIES[i] for i in cls_ids]
counts    = [class_counts[i] for i in cls_ids]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(cls_names, counts, color="steelblue", edgecolor="white")

# Annotate bars
for bar, cnt in zip(bars, counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 500,
        f"{cnt:,}",
        ha="center", va="bottom", fontsize=9
    )

ax.set_yscale("log")
ax.set_ylabel("Instance count (log scale)")
ax.set_title("AU-AIR Class Distribution (severe imbalance: Car dominates)")
ax.set_xlabel("Class")
plt.tight_layout()
plt.savefig("class_distribution.png", dpi=150)
plt.show()

As we can see above, classes are not equally distributed, but it is normal when we deal with object detection tasks, especially from fisheye, UAV-based scene images. Because vehicles like motorbikes, buses, bicycles are really few againts to the car, trucks, etc.

SO: The AU-AIR dataset is severely imbalanced, with the Car class dominating the annotations. This imbalance is expected to bias the detector toward frequent classes and make learning minority categories more difficult, which should be considered when interpreting per-class AP results.

### Bounding Box Size Analysis - Do we need for SAHI?

What is SAHI?

SAHI = Slicing Aided Hyper Inference

It's a technique designed for detecting "small objects" in high-resolution images. Here's the core idea: Instead of feeding the full 1920x1080 image into the model (which gets downscaled to 640x640, making small objects tiny or invisible), SAHI:

* Slices the full image into overlapping 640x640 tiles
* Runs the detector on each tile independently (each tile is at full resolution)
* Merges all detections back using NMS (non-max suppression) to remove duplicates

First, we need to understand whether we need to use SAHI? Do we have really small objects in the images? When we look at the dataset, we can see objects that are far away from the camera objective; they are much smaller than the objects (vehicles) that are closer to the drone. Let's analyze it by checking available bounding boxes.

You will see the COCO definition for in which sizes we will say it is small object, medium etc.

These threshold are directly obtained by official pycocotools evaluation code --> cocoeval.py

https://github.com/cocodataset/cocoapi/blob/master/PythonAPI/pycocotools/cocoeval.py

In [ ]:
"""class Params:
    '''
    Params for coco evaluation api
    '''
    def setDetParams(self):
        self.imgIds = []
        self.catIds = []
        # np.arange causes trouble.  the data point on arange is slightly larger than the true value
        self.iouThrs = np.linspace(.5, 0.95, int(np.round((0.95 - .5) / .05)) + 1, endpoint=True)
        self.recThrs = np.linspace(.0, 1.00, int(np.round((1.00 - .0) / .01)) + 1, endpoint=True)
        self.maxDets = [1, 10, 100]
        self.areaRng = [[0 ** 2, 1e5 ** 2], [0 ** 2, 32 ** 2], [32 ** 2, 96 ** 2], [96 ** 2, 1e5 ** 2]]
        self.areaRngLbl = ['all', 'small', 'medium', 'large']
        self.useCats = 1"""

In [ ]:
# Collect ALL bbox dimensions (in absolute pixels)
bbox_widths  = []
bbox_heights = []
bbox_areas   = []
# Per-class area lists for box plots
per_class_areas = defaultdict(list)

SMALL_THRESH = 32 * 32   # COCO definition: area < 32 px
MEDIUM_THRESH = 96 * 96  # COCO medium: 32-96 px

for ann in annotations:
    img_w = ann["image_width:"]   # note the typo colon in the key
    img_h = ann["image_height"]
    for bbox in ann["bbox"]:
        w = bbox["width"]
        h = bbox["height"]
        area = w * h
        bbox_widths.append(w)
        bbox_heights.append(h)
        bbox_areas.append(area)
        per_class_areas[bbox["class"]].append(area)

bbox_areas_np = np.array(bbox_areas)

n_small  = (bbox_areas_np < SMALL_THRESH).sum()
n_medium = ((bbox_areas_np >= SMALL_THRESH) & (bbox_areas_np < MEDIUM_THRESH)).sum()
n_large  = (bbox_areas_np >= MEDIUM_THRESH).sum()

print("Bounding Box Size Statistics (absolute pixels)")
print("=" * 50)
print(f"  Mean  area : {np.mean(bbox_areas_np):>10,.1f} px")
print(f"  Median area: {np.median(bbox_areas_np):>10,.1f} px")
print(f"  Min   area : {np.min(bbox_areas_np):>10,.1f} px")
print(f"  Max   area : {np.max(bbox_areas_np):>10,.1f} px")
print()
print(f"  Mean  width : {np.mean(bbox_widths):>8.1f} px")
print(f"  Mean  height: {np.mean(bbox_heights):>8.1f} px")
print()
print(f"COCO size categories (out of {total_bboxes:,} total):")
print(f"  Small  (area < 32^2)          : {n_small:>8,}  ({100*n_small/total_bboxes:.1f}%)")
print(f"  Medium (area > 32^2 and < 96^2)    : {n_medium:>8,}  ({100*n_medium/total_bboxes:.1f}%)")
print(f"  Large  (area > 96^2)          : {n_large:>8,}  ({100*n_large/total_bboxes:.1f}%)")

* Look at how many bounding boxes of the classes are classified as "small"

In [ ]:
# Per-class area statistics (mean and median)
print(f"{'Class':<12} {'Mean area':>12} {'Median area':>12} {'% small':>8}")
print("-" * 50)
for cls_id in sorted(per_class_areas.keys()):
    areas = np.array(per_class_areas[cls_id])
    pct_small = 100 * (areas < SMALL_THRESH).sum() / len(areas)
    print(f"{CATEGORIES[cls_id]:<12} {np.mean(areas):>12,.0f} {np.median(areas):>12,.0f} {pct_small:>7.1f}%")

We're seeing small objects are more common in classes such as human, car, motorbike, and bicycle, while truck, van, and bus generally appear larger. This suggests that small-object detection is an important challenge in AU-AIR, especially for visually small and less frequent categories.

We also see lots of cars in small dimension is probably caused by cars in the parking area or occlusion caused by consecutive cars etc.

In [ ]:
# Lets look at area distribution histogram + small-object threshold
# Put small-mid thresholds as well
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: histogram of sqrt(area) (effective side length)
ax = axes[0]
sqrt_areas = np.sqrt(bbox_areas_np)
ax.hist(sqrt_areas, bins=80, color="steelblue", edgecolor="none", alpha=0.85)
ax.axvline(32, color="red",    linestyle="--", linewidth=1.5, label="Small threshold (32 px)") # Put small
ax.axvline(96, color="orange", linestyle="--", linewidth=1.5, label="Medium threshold (96 px)") # Put mid
ax.set_xlabel("Effective side length sqrt(area) (pixels)")
ax.set_ylabel("Count")
ax.set_title("BBox size distribution - AU-AIR")
ax.legend(fontsize=9)

# Right: pie chart small / medium / large
ax = axes[1]
sizes  = [n_small, n_medium, n_large]
labels = [
    f"Small\n{100*n_small/total_bboxes:.1f}%",
    f"Medium\n{100*n_medium/total_bboxes:.1f}%",
    f"Large\n{100*n_large/total_bboxes:.1f}%",
]
colors = ["#d62728", "#ff7f0e", "#2ca02c"]
ax.pie(sizes, labels=labels, colors=colors, startangle=90,
       wedgeprops={"edgecolor": "white", "linewidth": 1.5})
ax.set_title("Object size categories (COCO definition)")

plt.show()

* bicycle, motorbike, human, and car have the smallest median bounding-box areas, so they are likely harder to detect.

In [ ]:
# Per-class median bbox area (log scale) shows which classes are smallest
cls_median_areas = {
    CATEGORIES[cls_id]: np.median(areas)
    for cls_id, areas in per_class_areas.items()
}
sorted_cls = sorted(cls_median_areas, key=cls_median_areas.get)
sorted_vals = [cls_median_areas[c] for c in sorted_cls]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(sorted_cls, sorted_vals, color="steelblue", edgecolor="white")
ax.axvline(SMALL_THRESH,  color="red",    linestyle="--", linewidth=1.2, label="Small")
ax.axvline(MEDIUM_THRESH, color="orange", linestyle="--", linewidth=1.2, label="Medium")
ax.set_xscale("log")
ax.set_xlabel("Median bbox area (px, log scale)")
ax.set_title("Per-class median bbox area smaller = harder to detect")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("per_class_bbox_area.png", dpi=150)
plt.show()

### Summary & Motivation for SAHI

* **Car** accounts for ~77% of instances --> *Class imbalance is an important topic when the evaluation is made as well. Let's look at it again at the end of the task*
* **Motorbike** has only 319 instances --> *Class imbalance is an important topic when the evaluation is made as well. Let's look at it again at the end of the task*
* A significant percentages of bboxes are **small** (area < 32^2)  ---> *apprx 12%*
* Images are **1920x1080**, when the model input is very low, heavy downscaling loses small-object detail (where we need to use SAHI)
* **SAHI** slices the full-res image into 640x640 tiles --> Each tile keeps original resolution small objects become detectable

## Step 3: Train / Val / Test Split

We need to plit by whole video clip not by individual frame

Consecutive frames in a UAV video are nearly identical (same scene, slightly shifted).
Splitting randomly by frame would cause data leakage: the model sees nearly the same
scene in both train and test, inflating accuracy. Assigning whole clips to one split
guarantees zero temporal leakage.

Target: ~85% train, ~8% val, ~7% test (It is same with the AU-AIR paper)

In [ ]:
# Filename pattern:  frame_<timestamp>_x_<framenum>.jpg
#                    frame_<timestamp>_xx_<framenum>.jpg
# We split on "_x" and take the first part the timestamp = clip ID

def get_clip_id(image_name):
    return image_name.replace("frame_", "").split("_x")[0]

# Check with example
test_names = [
    "frame_20190829091111_x_0000156.jpg",
    "frame_20190905091750_xx_0001973.jpg",
    "frame_20190905111947_x_0000001.jpg",
]
for name in test_names:
    print(f"{name}  -->> clip: {get_clip_id(name)}")

In [ ]:
# Count frames per clip to be sure

clip_frame_counts = Counter()
for ann in annotations:
    clip_frame_counts[get_clip_id(ann["image_name"])] += 1

total_frames = sum(clip_frame_counts.values())

print(f"{'Clip ID':<20} {'Frames':>8} {'Cum. frames':>12} {'Cum. %':>8}")
print("-" * 55)
cumulative = 0
for clip_id, count in sorted(clip_frame_counts.items(), key=lambda x: -x[1]):
    cumulative += count
    print(f"{clip_id:<20} {count:>8,} {cumulative:>12,} {100*cumulative/total_frames:>7.1f}%")

In [ ]:
# Assign clips to splits
# It is a manual operation

# Train: 5 largest clips 27,880 frames --> 84.9%
# Val: 1 clip  2,592 frames --> 7.9%
# Test: 2 smallest clips 2,351 frames --> 7.2%
#
# This matches the AU-AIR paper's reported apprx 30K train-val / apprx 2,823 test

CLIP_SPLIT = {
    # train
    "20190906150731": "train",
    "20190905103112": "train",
    "20190905091750": "train",
    "20190905112522": "train",
    "20190905142119": "train",
    # val
    "20190829091111": "val",
    # test
    "20190905143505": "test",
    "20190905111947": "test",
}

# Attach split label to every annotation
for ann in annotations:
    ann["split"] = CLIP_SPLIT[get_clip_id(ann["image_name"])]

# LOOK AT ANNOTATION DIST AS WELL FOR EACH DATASET TRAIN-TEST-VAL
split_frames = Counter(ann["split"] for ann in annotations)
split_bboxes = Counter()
for ann in annotations:
    split_bboxes[ann["split"]] += len(ann["bbox"])

print(f"{'Split':<8} {'Frames':>8} {'Frames %':>10} {'BBoxes':>10} {'BBoxes %':>10}")
print("-" * 52)
for split in ["train", "val", "test"]:
    f  = split_frames[split]
    b  = split_bboxes[split]
    fp = 100 * f / total_frames
    bp = 100 * b / total_bboxes
    print(f"{split:<8} {f:>8,} {fp:>9.1f}% {b:>10,} {bp:>9.1f}%")
print("-" * 52)
print(f"{'TOTAL':<8} {total_frames:>8,} {'100.0%':>10} {total_bboxes:>10,} {'100.0%':>10}")

Perfect!

## Step 4: Format Conversion AU-AIR ---> YOLO

We'll use RT-DETR, it only accepts YOLO format. It cannot read AU-AIR's JSON directly.

AU-AIR bbox format (absolute pixels): top, left, height, width, class


YOLO format requires:

* One .txt file per image (same name as the image)
* Each line = one object: class x_center y_center width height
* All coordinates normalized 0–1
* A .yaml file telling the model where the folders are

**YOLO bbox format** (normalized relative to image size):class  x_center  y_center  width  height

**Conversion:**
```
x_center = (left + width/2)  / image_width
y_center = (top  + height/2) / image_height
w_norm   = width             / image_width
h_norm   = height            / image_height
```

All values are converted to [0, 1] to guard against annotation boundary errors.

In [ ]:
# Create directory structure

for split in ['train', 'val', 'test']:
    (YOLO_ROOT / 'images' / split).mkdir(parents=True, exist_ok=True)
    (YOLO_ROOT / 'labels' / split).mkdir(parents=True, exist_ok=True)

print('Directory structure created at:', YOLO_ROOT)
for p in sorted(YOLO_ROOT.rglob('*')):
    if p.is_dir():
        print(f'  {p.relative_to(YOLO_ROOT.parent)}')


In [ ]:
def auair_to_yolo(bbox, img_w, img_h):
    """
    Convert a single AU-AIR bbox dict to a YOLO label string.

    AU-AIR keys : top, left, height, width, class  (absolute pixels)
    YOLO format : class x_center y_center width height  (normalized 0-1)
    """
    x_center = (bbox["left"] + bbox["width"]  / 2) / img_w
    y_center = (bbox["top"]  + bbox["height"] / 2) / img_h
    w_norm   =  bbox["width"]                      / img_w
    h_norm   =  bbox["height"]                     / img_h

    # Convert to [0, 1], guards against bboxes that slightly exceed image border
    x_center = max(0.0, min(1.0, x_center))
    y_center = max(0.0, min(1.0, y_center))
    w_norm   = max(0.0, min(1.0, w_norm))
    h_norm   = max(0.0, min(1.0, h_norm))

    return f"{bbox['class']} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}"

We symlinked images and generated YOLO label files to create a training-ready dataset structure for Ultralytics without duplicating the original image data. (I asked it AI tool and it suggested this way)

In [ ]:
# Main loop: symlink images + write label files
# IMG_DIR is set by kaggle input
import os

skipped   = 0
processed = 0

for ann in annotations:
    img_name = ann['image_name']
    split    = ann['split']
    img_w    = ann['image_width:']   # note the typo colon key, it includes :???
    img_h    = ann['image_height']

    # Image: symlink into auair_yolo/images/{split}/
    src = IMG_DIR / img_name
    dst = YOLO_ROOT / 'images' / split / img_name

    if not src.exists():
        skipped += 1
        continue

    if not dst.exists():
        os.symlink(src.resolve(), dst)  # absolute symlink, it works on Kaggle

    # Label: write .txt file into auair_yolo/labels/{split}/
    label_path = YOLO_ROOT / 'labels' / split / (Path(img_name).stem + '.txt')
    if not label_path.exists():
        lines = [auair_to_yolo(bbox, img_w, img_h) for bbox in ann['bbox']]
        label_path.write_text('\n'.join(lines))

    processed += 1

print(f'Done.')
print(f'  Processed : {processed:,} images')
print(f'  Skipped   : {skipped:,}  (source file missing)')


In [ ]:
# Write auair.yaml
yaml_content = {
    "path"  : str(YOLO_ROOT),
    "train" : "images/train",
    "val"   : "images/val",
    "test"  : "images/test",
    "nc"    : 8,
    "names" : {
        0: "Human",
        1: "Car",
        2: "Truck",
        3: "Van",
        4: "Motorbike",
        5: "Bicycle",
        6: "Bus",
        7: "Trailer",
    },
}

yaml_path = YOLO_ROOT / "auair.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(yaml_content, f, default_flow_style=False, sort_keys=False)

print("auair.yaml written to:", yaml_path)
print()
print(yaml_path.read_text())

In [ ]:
# Verify to be sure this operation works that we intended
expected = {"train": 27_880, "val": 2_592, "test": 2_351}

print(f"{'Split':<8} {'Images':>8} {'Labels':>8} {'Expected':>10} {'OK?':>6}")
print("-" * 46)

all_ok = True
for split in ["train", "val", "test"]:
    n_images = len(list((YOLO_ROOT / "images" / split).iterdir()))
    n_labels = len(list((YOLO_ROOT / "labels" / split).iterdir()))
    exp      = expected[split]
    ok       = "Yes" if n_images == exp == n_labels else "No"
    if ok == "No":
        all_ok = False
    print(f"{split:<8} {n_images:>8,} {n_labels:>8,} {exp:>10,} {ok:>6}")

print()
if all_ok:
    print("All splits verified, ready for training.")
else:
    print("Mismatch detected, check skipped count above.")

Perfect!!!!

---
## Step 5: RT-DETR Fine-Tuning

### What is RT-DETR?

RT-DETR (Real-Time Detection Transformer) is a transformer-based object detector from Baidu (2023). We'll use it from Ultralytics which is the software library we used to train and evaluate the detector.

### Why we chose RT-DETR?
We chose RT-DETR which is a transformer detector that is practical to fine-tune. It combines convolutional feature extraction with a transformer decoder, so it can model global context better than conventional CNN-only detectors. This is useful for the AU-AIR dataset, where objects are small, dense, and sometimes hard to separate from the background. Another reason for choosing RT-DETR is that it is supported by Ultralytics, which makes training, evaluation, and comparison easier in a reproducible notebook setting.

### Why it is suitable for this dataset?
AU-AIR contains many small objects, especially cars, humans, bicycles, and motorbikes seen from a UAV viewpoint. In such scenes, attention mechanisms can help the model focus on relevant regions and capture long-range relationships across the image. RT-DETR is also attractive because it is an end-to-end detector and does not rely on traditional non-maximum suppression in the same way as YOLO-style models. Since we fine-tune a pretrained RT-DETR model rather than training from scratch, we can benefit from already learned visual features while adapting the detector to the 8 AU-AIR classes.

### Possible alternatives
Other transformer-based alternatives could also be used.
* One option is the original DETR, which is important historically but is slower to train and usually less practical.
* Deformable DETR is another strong alternative because it improves convergence and handles small objects better than vanilla DETR.
* DINO is also a powerful transformer detector with strong performance, but it is more complex and heavier for this task.
 * We selected RT-DETR because it offers a good balance between transformer-based design, implementation simplicity, training speed, and strong detection performance.


#### Key differences from YOLO:
- Architecture: YOLO is CNN-based; RT-DETR uses a transformer decoder.
- Post-processing: YOLO uses NMS; RT-DETR is NMS-free.
- Context: RT-DETR captures global context better.
- Small objects: RT-DETR can work better on small, dense objects.
- Tradeoff: YOLO is simpler; RT-DETR is more attention-based.


We use **RT-DETR-L** pretrained on COCO — fine-tuning keeps the learned visual
features and only adapts the detection head to our 8 AU-AIR classes.

### Hyperparameter choices

- imgsz = 640: standard RT-DETR input size and matches the SAHI tile size.
- epochs = 25: enough training time to exceed YOLO, but it can require more maybe, but it is computationally intensive and requires more resource on Kaggle.
- batch = 24: due to computation time.
- lr0 = 1e-4: low initial learning rate for stable fine-tuning.
- lrf = 0.01: gradually reduces the learning rate by the end of training.
- warmup_epochs = 3: avoids unstable updates at the beginning.
- patience = 15: stops training if validation mAP does not improve.
- seed = 42: keeps the experiment reproducible.

In [ ]:
from ultralytics import RTDETR

model = RTDETR('rtdetr-l.pt')
print(model.info())

In [ ]:
# Fine-tune on AU-AIR
import torch
from pathlib import Path
from ultralytics import RTDETR

# Safety fallbacks
if 'DEVICE' not in dir():
    DEVICE = 0 if torch.cuda.is_available() else 'cpu'
if 'PROJECT_ROOT' not in dir():
    PROJECT_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('..').resolve()
if 'YOLO_ROOT' not in dir():
    YOLO_ROOT = PROJECT_ROOT / 'auair_yolo'

# TRAINING MODE!!!!
# SKIP_TRAINING = True  → skip training entirely, load a saved checkpoint below
# RESUME        = True  → continue training from last.pt (a previous run) --> to resume on Kaggle

SKIP_TRAINING = False
RESUME        = False   # set True on 2nd+ run to add more epochs on top

if SKIP_TRAINING:
    print('SKIP_TRAINING=True — skipping training, loading checkpoint in Step 6.')
else:
    N_GPUS       = torch.cuda.device_count()
    DEVICE_TRAIN = '0,1' if N_GPUS >= 2 else 0
    print(f'GPUs available: {N_GPUS}  →  device={DEVICE_TRAIN}')

    if RESUME:
        resume_candidates = [
            Path('/kaggle/working/runs/rtdetr_auair/weights/last.pt'),
            *Path('/kaggle/input').rglob('last.pt'),
        ]
        resume_pt = next((p for p in resume_candidates if p.exists()), None)
        if resume_pt is None:
            raise FileNotFoundError(
                'last.pt not found. Upload the previous rtdetr_results.zip '
                'as a Kaggle input dataset before setting RESUME=True.'
            )
        print(f'Resuming from: {resume_pt}')
        model = RTDETR(str(resume_pt))
    else:
        model = RTDETR('rtdetr-l.pt')

    results = model.train(
        data         = str(YOLO_ROOT / 'auair.yaml'),
        epochs       = 25,
        imgsz        = 640,
        batch        = 24,
        lr0          = 1e-4,
        lrf          = 0.01,
        warmup_epochs= 3,
        patience     = 15,
        device       = DEVICE_TRAIN,
        project      = str(PROJECT_ROOT / 'runs'),
        name         = 'rtdetr_auair',
        seed         = 42,
        exist_ok     = True,
        workers      = 4,
    ) # Ultralytics automatically trains on train and evaluates after epochs on val


In [ ]:
from pathlib import Path

# Check working directory — training saves here in the current session
working_matches = list(Path('/kaggle/working').rglob('best.pt'))
print("In working dir:", working_matches)

In [ ]:
# Search for results.csv
csv_matches = list(Path('/kaggle/working').rglob('results.csv'))
print("results.csv:", csv_matches)

# Also list everything in the run folder
run_dir = Path('/kaggle/working/runs/rtdetr_auair')
print("\nAll files in run dir:")
for f in sorted(run_dir.rglob('*')):
    if f.is_file():
        print(f" {f}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

csv_path = Path('/kaggle/working/runs/rtdetr_auair/results.csv')
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()   # remove whitespace

# Print actual column names first
print(df.columns.tolist())

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# Display the auto-generated results.png
img = Image.open('/kaggle/working/runs/rtdetr_auair/results.png')
plt.figure(figsize=(16, 8))
plt.imshow(img)
plt.axis('off')
plt.title('Training Results')
plt.tight_layout()
plt.show()

In [ ]:
for fname in ['confusion_matrix_normalized.png', 'BoxPR_curve.png', 'BoxF1_curve.png']:
    img = Image.open(f'/kaggle/working/runs/rtdetr_auair/{fname}')
    plt.figure(figsize=(10, 6))
    plt.imshow(img); plt.axis('off'); plt.title(fname)
    plt.show()

In [ ]:
import shutil

shutil.make_archive(
    '/kaggle/working/rtdetr_results',  # output zip name
    'zip',
    '/kaggle/working/runs/rtdetr_auair'  # folder to zip
)
print("Done → download rtdetr_results.zip from Output panel")

## Step 6: Standard Evaluation (Without SAHI)

Run the built-in Ultralytics on the test split.
This gives us the **RT-DETR baseline**.
We'll compare this directly against RT-DETR + SAHI in the next step.


In [ ]:
# Load best checkpoint
from ultralytics import RTDETR
from pathlib import Path

if 'PROJECT_ROOT' not in dir():
    PROJECT_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('..').resolve()
if 'YOLO_ROOT' not in dir():
    YOLO_ROOT = PROJECT_ROOT / 'auair_yolo'

# Auto-find best.pt: check current session first, then saved input
candidates = [
    Path('/kaggle/working/runs/rtdetr_auair/weights/best.pt'),   # current session
    *Path('/kaggle/input').rglob('best.pt'),                      # saved version
]
BEST_PT = next((p for p in candidates if p.exists()), None)

if BEST_PT is None:
    raise FileNotFoundError('best.pt not found. Add saved output as Input dataset.')

print('Loading checkpoint:', BEST_PT)
model = RTDETR(str(BEST_PT))


Remember:

* mAP@50: mean Average Precision at IoU = 0.50 --> A prediction is counted as correct if it overlaps the ground-truth box by at least 50%.

* mAP@50-95: mean Average Precision averaged over multiple IoU thresholds from 0.50 to 0.95 with step 0.05 --> This is a stricter and more comprehensive metric.


In [24]:
# Evaluate on test split
metrics = model.val(
    data  = str(YOLO_ROOT / 'auair.yaml'),
    split = 'test',
    imgsz = 640,
    batch = 8,
    device= DEVICE if 'DEVICE' in dir() else 0,
)

print('\n── Test Results (RT-DETR, no SAHI) ──')
print(f'  mAP@50      : {metrics.box.map50:.4f}')
print(f'  mAP@50-95   : {metrics.box.map:.4f}')


In [ ]:
# Lets look at per-class AP@50

CATEGORIES = ['Human','Car','Truck','Van','Motorbike','Bicycle','Bus','Trailer']

per_class_ap50 = metrics.box.ap50          # numpy array, one value per class
map50_no_sahi  = float(metrics.box.map50)

print(f'{'Class':<12} {'AP@50':>8}')
print('-' * 22)
for cls, ap in zip(CATEGORIES, per_class_ap50):
    print(f'{cls:<12} {ap*100:>7.2f}%')
print('-' * 22)
print(f'{'mAP@50':<12} {map50_no_sahi*100:>7.2f}%')


## Step 7: SAHI Integration

Wrap the trained RT-DETR with SAHI's sliced inference.
Each 1920×1080 image is sliced into overlapping 640×640 tiles;
the model runs on each tile and detections are merged back.


In [ ]:
# Model with SAHI
from sahi import AutoDetectionModel

DEVICE_STR = f'cuda:{DEVICE}' if isinstance(DEVICE, int) else DEVICE

# confidence_threshold=0.4
#   SAHI runs the model independently on each of 8 tiles per 1920×1080 image.
#   SAHI's post-processing NMS cannot fully suppress all duplicates → many false positives → Car AP drops.
#   0.4 keeps only confident predictions, producing cleaner tile-merge results.
detection_model = AutoDetectionModel.from_pretrained(
    model_type          = 'ultralytics',
    model_path          = str(BEST_PT),
    confidence_threshold= 0.4,
    device              = DEVICE_STR,
)
print('SAHI model loaded.')
print(f'  Device              : {DEVICE_STR}')
print(f'  Confidence threshold: 0.4')

SAHI_PARAMS = dict(
    slice_height             = 640,   # matches training resolution
    slice_width              = 640,   # matches training resolution
    overlap_height_ratio     = 0.2,   # 128px overlap — enough for most objects at UAV altitude
    overlap_width_ratio      = 0.2,   # same horizontally; gives 8 tiles per 1920×1080 frame
    perform_standard_pred    = True,  # also run on full image → catches large objects
    postprocess_type         = "GREEDYNMM",   # Greedy Non-Maximum Merging: better than plain
                                              # NMS for tile-stitching — merges overlapping
                                              # detections instead of just suppressing them
    postprocess_match_metric = "IOS",         # Intersection over Smaller area: more suitable
                                              # than IoU when a small tile-bbox overlaps a
                                              # full-image bbox of very different size
    postprocess_match_threshold = 0.5,        # merge detections that overlap >50%
)
print(f'\nSAHI inference parameters:')
for k, v in SAHI_PARAMS.items():
    print(f'  {k:<30}: {v}')


In [ ]:
# Check SAHI on a single image
from sahi.predict import get_sliced_prediction
import random

if 'annotations' not in dir():
    import json as _json
    with open(ANN_PATH) as f:
        _raw = _json.load(f)
    annotations = _raw['annotations']

test_anns  = [a for a in annotations if a.get('split') == 'test']
sample_ann = random.Random(42).choice(test_anns)
sample_img = str(IMG_DIR / sample_ann['image_name'])

result = get_sliced_prediction(sample_img, detection_model, **SAHI_PARAMS, verbose=1)
print(f'Image           : {sample_ann["image_name"]}')
print(f'GT boxes        : {len(sample_ann["bbox"])}')
print(f'SAHI detections : {len(result.object_prediction_list)}')


SAHI may produce more detections than the number of ground-truth boxes because sliced inference can introduce duplicate or extra candidate detections. This is acceptable at this stage; the real criterion is whether these predictions improve final precision-recall performance after evaluation.

SO:
- more predicted boxes than GT is not inherently bad
- but if many of them do not match GT, precision will drop

## Step 8: Full Test Set Evaluation with SAHI

SAHI bypasses Ultralytics' validation pipeline, so we compute mAP manually
using Pascal VOC 11-point interpolation to match the AU-AIR paper.

**Pascal VOC 11-point interpolation** computes AP by averaging the best precision values at 11 fixed recall levels from 0.0 to 1.0.

For each class:
1. Collect all predictions sorted by confidence
2. Match to GT boxes using IoU ≥ 0.5
3. Compute precision-recall curve
4. Average precision at 11 recall thresholds [0, 0.1, …, 1.0]

BRIEFLY:
* SAHI slices each test image into overlapping patches.
* The trained RT-DETR model runs on each patch.
* This can produce multiple detections for the same object, especially near slice overlaps.
* SAHI then merges and filters those detections to produce final boxes on the original image.


In [ ]:
import numpy as np

def compute_iou(box1, box2):
    """IoU of two [x1,y1,x2,y2] boxes."""
    xi1 = max(box1[0], box2[0]); yi1 = max(box1[1], box2[1])
    xi2 = min(box1[2], box2[2]); yi2 = min(box1[3], box2[3])
    inter = max(0, xi2-xi1) * max(0, yi2-yi1)
    a1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    a2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    return inter / (a1 + a2 - inter + 1e-6)

def compute_ap_11point(precisions, recalls):
    """Pascal VOC 11-point interpolated AP."""
    ap = 0.0
    for thr in np.linspace(0, 1, 11):
        p = precisions[recalls >= thr]
        ap += (p.max() if p.size > 0 else 0.0)
    return ap / 11


In [ ]:
# Run SAHI on full test set + collect predictions
from sahi.predict import get_sliced_prediction
from collections import defaultdict

if 'test_anns' not in dir():
    test_anns = [a for a in annotations if a.get('split') == 'test']

predictions = defaultdict(list)
gt_boxes    = defaultdict(list)

print(f'Running SAHI on {len(test_anns):,} test images...')
for idx, ann in enumerate(test_anns):
    img_path = str(IMG_DIR / ann['image_name'])
    img_id   = ann['image_name']

    # Ground truth
    for b in ann['bbox']:
        x1 = b['left'];            y1 = b['top']
        x2 = b['left']+b['width']; y2 = b['top']+b['height']
        gt_boxes[b['class']].append((img_id, [x1, y1, x2, y2]))

    # SAHI prediction
    result = get_sliced_prediction(img_path, detection_model, **SAHI_PARAMS, verbose=0)
    for pred in result.object_prediction_list:
        cls  = pred.category.id
        conf = pred.score.value
        b    = pred.bbox
        predictions[cls].append((conf, img_id, [b.minx, b.miny, b.maxx, b.maxy]))

    if (idx+1) % 200 == 0:
        print(f'  {idx+1}/{len(test_anns)} done')

print('SAHI inference complete.')


In [ ]:
# Compute per-class AP and mAP
CATEGORIES = ['Human','Car','Truck','Van','Motorbike','Bicycle','Bus','Trailer']
IOU_THRESH  = 0.5
ap_per_class_sahi = {}

for cls_id, cls_name in enumerate(CATEGORIES):
    preds = sorted(predictions[cls_id], key=lambda x: -x[0])  # sort by conf desc
    gts   = gt_boxes[cls_id]
    n_gt  = len(gts)

    if n_gt == 0:
        ap_per_class_sahi[cls_name] = 0.0
        continue

    # Track which GT boxes are already matched
    matched = set()
    tp = np.zeros(len(preds))
    fp = np.zeros(len(preds))

    for i, (conf, img_id, pred_box) in enumerate(preds):
        # Find GT boxes in the same image
        img_gts = [(j, g[1]) for j, g in enumerate(gts) if g[0] == img_id]
        best_iou, best_j = 0, -1
        for j, gt_box in img_gts:
            iou = compute_iou(pred_box, gt_box)
            if iou > best_iou:
                best_iou, best_j = iou, j
        if best_iou >= IOU_THRESH and best_j not in matched:
            tp[i] = 1; matched.add(best_j)
        else:
            fp[i] = 1

    tp_cum = np.cumsum(tp)
    fp_cum = np.cumsum(fp)
    recalls    = tp_cum / (n_gt + 1e-6)
    precisions = tp_cum / (tp_cum + fp_cum + 1e-6)
    ap_per_class_sahi[cls_name] = compute_ap_11point(precisions, recalls)

map_sahi = np.mean(list(ap_per_class_sahi.values()))

print('── Per-class AP@50 (RT-DETR + SAHI) ──')
for cls_name, ap in ap_per_class_sahi.items():
    print(f'  {cls_name:<12} {ap*100:>6.2f}%')
print(f'  {"mAP":<12} {map_sahi*100:>6.2f}%')


## Step 9: Results Comparison Table

Compare RT-DETR + SAHI against the paper's baselines.


In [ ]:
# Comparison table
import pandas as pd

CATEGORIES = ['Human','Car','Truck','Van','Motorbike','Bicycle','Bus','Trailer']

# Baseline numbers from AU-AIR paper
yolov3_ap    = [34.05, 36.30, 47.13, 41.47,  4.80, 12.34, 51.78, 13.95]
mobilenet_ap = [22.86, 19.65, 34.74, 25.73,  0.01,  0.01, 39.63, 13.38]

rtdetr_no_sahi = [v*100 for v in per_class_ap50]
rtdetr_sahi    = [ap_per_class_sahi[c]*100 for c in CATEGORIES]

df = pd.DataFrame({
    'Class'            : CATEGORIES,
    'YOLOv3-Tiny'      : yolov3_ap,
    'MobileNetV2-SSDLite': mobilenet_ap,
    'RT-DETR (no SAHI)': rtdetr_no_sahi,
    'RT-DETR + SAHI'   : rtdetr_sahi,
})

# Add mAP row
map_row = pd.DataFrame([{
    'Class'            : 'mAP',
    'YOLOv3-Tiny'      : 30.22,
    'MobileNetV2-SSDLite': 19.50,
    'RT-DETR (no SAHI)': map50_no_sahi*100,
    'RT-DETR + SAHI'   : map_sahi*100,
}])
df = pd.concat([df, map_row], ignore_index=True)
df.set_index('Class', inplace=True)
print(df.round(2).to_string())


In [ ]:
#Grouped bar chart

import matplotlib.pyplot as plt
import numpy as np

models  = ['YOLOv3-Tiny', 'MobileNetV2-SSDLite', 'RT-DETR (no SAHI)', 'RT-DETR + SAHI']
colors  = ['#4878d0', '#ee854a', '#6acc65', '#d65f5f']
x       = np.arange(len(CATEGORIES))
width   = 0.2

fig, ax = plt.subplots(figsize=(14, 5))
for i, (model_name, color) in enumerate(zip(models, colors)):
    vals = df.loc[CATEGORIES, model_name].values.astype(float)
    ax.bar(x + i*width, vals, width, label=model_name, color=color, alpha=0.85)

ax.set_xticks(x + width*1.5)
ax.set_xticklabels(CATEGORIES)
ax.set_ylabel('AP@50 (%)')
ax.set_title('Per-class AP@50 — Model Comparison')
ax.legend(loc='upper right', fontsize=9)
ax.axhline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.savefig('comparison_chart.png', dpi=150)
plt.show()


## Step 10: Final Sample Visualization

Ground Truth (left) vs RT-DETR + SAHI predictions (right)
for 4 test images.


In [ ]:
# Visualize GT vs SAHI predictions
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import random

CATEGORIES = ['Human','Car','Truck','Van','Motorbike','Bicycle','Bus','Trailer']
PALETTE = ['#e6194b','#3cb44b','#ffe119','#4363d8',
           '#f58231','#911eb4','#42d4f4','#f032e6']

def draw_gt(ax, img, bboxes):
    ax.imshow(img); ax.axis('off'); ax.set_title('Ground Truth', fontsize=8)
    for b in bboxes:
        c = b['class']
        rect = patches.Rectangle((b['left'], b['top']), b['width'], b['height'],
                                   linewidth=2, edgecolor=PALETTE[c], facecolor='none')
        ax.add_patch(rect)
        ax.text(b['left'], b['top']-4, CATEGORIES[c], fontsize=6, color='white',
                bbox=dict(facecolor=PALETTE[c], alpha=0.8, pad=1, edgecolor='none'))

def draw_preds(ax, img, preds):
    ax.imshow(img); ax.axis('off'); ax.set_title('RT-DETR + SAHI', fontsize=8)
    for pred in preds:
        c  = pred.category.id
        b  = pred.bbox
        sc = pred.score.value
        rect = patches.Rectangle((b.minx, b.miny), b.maxx-b.minx, b.maxy-b.miny,
                                   linewidth=2, edgecolor=PALETTE[c % 8], facecolor='none')
        ax.add_patch(rect)
        ax.text(b.minx, b.miny-4, f'{CATEGORIES[c]} {sc:.2f}', fontsize=6, color='white',
                bbox=dict(facecolor=PALETTE[c % 8], alpha=0.8, pad=1, edgecolor='none'))

samples = random.Random(42).sample(test_anns, 4)
fig, axes = plt.subplots(4, 2, figsize=(16, 18))

for row, ann in enumerate(samples):
    img = Image.open(IMG_DIR / ann['image_name']).convert('RGB')
    result = get_sliced_prediction(
        str(IMG_DIR / ann['image_name']), detection_model, **SAHI_PARAMS, verbose=0,
    )
    draw_gt   (axes[row][0], img, ann['bbox'])
    draw_preds(axes[row][1], img, result.object_prediction_list)

plt.suptitle('GT vs RT-DETR + SAHI', fontsize=12)
plt.tight_layout()
plt.savefig('final_sample_results.png', dpi=120, bbox_inches='tight')
plt.show()
